# 🔬 Movimiento Browniano: De Einstein a los Procesos Estocásticos

---

## Tabla de Contenidos

1. **Teoría de Einstein (1905):** Difusión y desplazamiento cuadrático medio
2. **Ecuación de Langevin:** Fluctuación-disipación y proceso de Wiener
3. **Ecuación de Fokker-Planck:** Derivación desde Chapman-Kolmogorov
4. **Tiempos de Primer Paso:** Problemas de frontera
5. **Proceso de Ornstein-Uhlenbeck:** Velocidad browniana
6. **Difusión Anómala:** Subdifusión, superdifusión y vuelos de Lévy
7. **Resumen y Referencias**

---

## Contexto Histórico

El **movimiento browniano** — el movimiento errático e incesante de partículas microscópicas suspendidas en un fluido — es uno de los fenómenos más fundamentales de la física estadística. Su estudio riguroso fue clave para confirmar la existencia de los átomos.

| Año | Protagonista | Contribución |
|------|-------------|-------------|
| **1827** | Robert Brown | Observa el movimiento irregular de granos de polen en agua |
| **1905** | Albert Einstein | Deriva $\langle x^2 \rangle = 2Dt$ desde la teoría cinético-molecular |
| **1906** | Marian Smoluchowski | Derivación independiente por métodos de caminata aleatoria |
| **1908** | Paul Langevin | Ecuación estocástica de movimiento: $m\ddot{x} = -\gamma\dot{x} + \eta(t)$ |
| **1908-09** | Jean Perrin | Verificación experimental, determina $N_A$. Nobel 1926 |
| **1931** | Andréi Kolmogórov | Formalización matemática rigurosa |

> *"En este trabajo se mostrará que, de acuerdo con la teoría cinético-molecular del calor, cuerpos de tamaño visible suspendidos en un líquido deben realizar movimientos observables..."*
> — A. Einstein, *Annalen der Physik* **17**, 549 (1905)

In [ ]:
# ============================================================
# Configuración e Importaciones
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

# Configuración de gráficos
plt.rcParams.update({
    'figure.figsize': (10, 7),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 100
})

print("✓ Librerías cargadas correctamente")

## 🧪 1. Teoría de Einstein (1905): Difusión y Desplazamiento Cuadrático Medio

### 1.1 El Artículo Fundacional

En 1905, Albert Einstein publicó *"Über die von der molekularkinetischen Theorie der Wärme geforderte Bewegung von in ruhenden Flüssigkeiten suspendierten Teilchen"*, donde derivó las propiedades del movimiento browniano **sin conocer las observaciones de Brown**, partiendo únicamente de la teoría cinético-molecular.

### 1.2 Derivación de Einstein

Einstein consideró una suspensión de partículas con densidad $\rho(x,t)$ y argumentó que:

1. **Cada partícula realiza una caminata aleatoria** independiente, con desplazamientos $\Delta$ en intervalos $\tau$.
2. La distribución de desplazamientos $\phi(\Delta)$ es simétrica: $\phi(\Delta) = \phi(-\Delta)$.

Expandiendo $\rho(x + \Delta, t)$ en serie de Taylor y promediando sobre $\phi(\Delta)$:

$$\rho(x, t+\tau) = \int_{-\infty}^{\infty} \rho(x+\Delta, t) \, \phi(\Delta) \, d\Delta \approx \rho(x,t) + \frac{\partial^2 \rho}{\partial x^2} \underbrace{\frac{\langle \Delta^2 \rangle}{2}}_{\text{segundo momento}}$$

Definiendo el **coeficiente de difusión**:

$$\boxed{D \equiv \frac{\langle \Delta^2 \rangle}{2\tau}}$$

se obtiene la **ecuación de difusión**:

$$\frac{\partial \rho}{\partial t} = D \frac{\partial^2 \rho}{\partial x^2}$$

### 1.3 Desplazamiento Cuadrático Medio

La solución para una partícula en el origen es gaussiana:

$$P(x,t) = \frac{1}{\sqrt{4\pi D t}} \exp\left(-\frac{x^2}{4Dt}\right)$$

El **desplazamiento cuadrático medio** (MSD) crece linealmente con el tiempo:

$$\boxed{\langle x^2(t) \rangle = 2Dt}$$

En $d$ dimensiones: $\langle r^2 \rangle = 2dDt$.

### 1.4 Relación de Stokes-Einstein

Einstein conectó $D$ con cantidades macroscópicas medibles:

$$\boxed{D = \frac{k_B T}{6\pi\mu a}}$$

donde $\mu$ es la viscosidad del fluido, $a$ el radio de la partícula, y $k_B$ la constante de Boltzmann. Esta relación permitió a **Jean Perrin** (1908) determinar el número de Avogadro $N_A$ experimentalmente, confirmando la existencia de los átomos.

In [ ]:
# ============================================================
# Derivación Simbólica: Teoría de Einstein del Movimiento Browniano
# ============================================================

import sympy as sp
from IPython.display import display, Math, Markdown

# Símbolos
x, t, D, k_B, T, eta, a, gamma_s = sp.symbols('x t D k_B T eta a gamma', positive=True)
P = sp.Function('P')

# --- 1. Ecuación de difusión ---
display(Markdown("### Ecuación de difusión"))
eq_difusion = sp.Eq(sp.Derivative(P(x, t), t), D * sp.Derivative(P(x, t), x, 2))
display(eq_difusion)

# --- 2. Solución fundamental (propagador gaussiano) ---
display(Markdown("### Solución fundamental $P(x,t)$ con $P(x,0) = \\delta(x)$"))
P_sol = 1 / sp.sqrt(4 * sp.pi * D * t) * sp.exp(-x**2 / (4 * D * t))
display(Math(f"P(x,t) = {sp.latex(P_sol)}"))

# Verificar que satisface la ecuación de difusión
dP_dt = sp.diff(P_sol, t)
d2P_dx2 = sp.diff(P_sol, x, 2)
check = sp.simplify(dP_dt - D * d2P_dx2)
display(Markdown(f"**Verificación:** $\\partial_t P - D\\partial_x^2 P = {sp.latex(check)}$ ✓"))

# --- 3. Desplazamiento cuadrático medio ---
display(Markdown("### Cálculo de $\\langle x^2 \\rangle$"))
msd_integral = sp.integrate(x**2 * P_sol, (x, -sp.oo, sp.oo))
msd_result = sp.simplify(msd_integral)
display(Math(f"\\langle x^2 \\rangle = \\int_{{-\\infty}}^{{\\infty}} x^2 P(x,t) \\, dx = {sp.latex(msd_result)}"))

# --- 4. Relación de Stokes-Einstein ---
display(Markdown("### Relación de Stokes-Einstein"))
D_einstein = k_B * T / (6 * sp.pi * eta * a)
display(Math(f"D = \\frac{{k_B T}}{{6\\pi\\eta a}} = {sp.latex(D_einstein)}"))

# --- 5. Cálculo numérico para una partícula de polen en agua ---
display(Markdown("### Ejemplo: Grano de polen ($a = 1\\,\\mu m$) en agua a $T = 300\\,K$"))
k_B_val = 1.381e-23    # J/K
T_val = 300             # K
eta_val = 1e-3          # Pa·s (agua)
a_val = 1e-6            # m (1 μm)

D_val = k_B_val * T_val / (6 * 3.14159 * eta_val * a_val)
print(f"D = {D_val:.4e} m²/s")
print(f"En t = 1 s:  √⟨x²⟩ = {(2*D_val*1)**0.5 * 1e6:.2f} μm")
print(f"En t = 60 s: √⟨x²⟩ = {(2*D_val*60)**0.5 * 1e6:.2f} μm")
print(f"\nEstos valores son consistentes con las observaciones de Perrin (1908).")

## ⚙️ 2. Ecuación de Langevin y Teorema de Fluctuación-Disipación

### 2.1 El Enfoque de Langevin (1908)

Paul Langevin propuso un enfoque complementario al de Einstein, escribiendo directamente una ecuación de movimiento estocástica para una partícula browniana:

$$\boxed{m\frac{dv}{dt} = -\gamma v + \eta(t)}$$

donde:
- $m$ es la masa de la partícula
- $\gamma$ es el **coeficiente de fricción** (fuerza de Stokes: $\gamma = 6\pi\mu a$ para una esfera de radio $a$)
- $\eta(t)$ es una **fuerza estocástica** (ruido blanco) que modela los impactos moleculares

### 2.2 Propiedades del Ruido Blanco

La fuerza estocástica $\eta(t)$ satisface:

$$\langle \eta(t) \rangle = 0 \qquad \text{(media nula)}$$

$$\langle \eta(t)\eta(t') \rangle = \Gamma \, \delta(t - t') \qquad \text{(incorrelacionado)}$$

donde $\Gamma$ es la **intensidad del ruido**, aún por determinar.

### 2.3 Solución de la Ecuación de Langevin

La ecuación de Langevin es una ODE lineal de primer orden para $v(t)$. Definiendo $\beta = \gamma/m$ (tasa de relajación):

$$v(t) = v_0 \, e^{-\beta t} + \frac{1}{m}\int_0^t e^{-\beta(t-t')} \eta(t') \, dt'$$

**Velocidad media:**
$$\langle v(t) \rangle = v_0 \, e^{-\beta t} \xrightarrow{t \gg 1/\beta} 0$$

**Desplazamiento cuadrático medio:** Langevin calculó directamente $\langle x^2 \rangle$ multiplicando la ecuación por $x$:

$$m\frac{d^2}{dt^2}\frac{\langle x^2 \rangle}{2} - m\langle v^2 \rangle = -\gamma \frac{d}{dt}\frac{\langle x^2 \rangle}{2}$$

Usando el **teorema de equipartición** $\frac{1}{2}m\langle v^2 \rangle = \frac{1}{2}k_B T$:

$$\langle x^2(t) \rangle = \frac{2k_B T}{\gamma}\left[t - \frac{m}{\gamma}\left(1 - e^{-\gamma t/m}\right)\right]$$

### 2.4 Límites Asintóticos

**Tiempos largos** ($t \gg m/\gamma$): régimen difusivo
$$\langle x^2 \rangle \approx \frac{2k_B T}{\gamma} t = 2Dt$$

recuperando el resultado de Einstein con la **relación de Einstein**:

$$\boxed{D = \frac{k_B T}{\gamma} = \frac{k_B T}{6\pi\mu a}}$$

**Tiempos cortos** ($t \ll m/\gamma$): régimen balístico
$$\langle x^2 \rangle \approx \frac{k_B T}{m} t^2 = \langle v^2 \rangle t^2$$

### 2.5 Teorema de Fluctuación-Disipación

Para que la distribución estacionaria sea la distribución de Maxwell-Boltzmann, la intensidad del ruido debe satisfacer:

$$\boxed{\Gamma = 2\gamma k_B T}$$

Esta es la **relación de fluctuación-disipación**: la intensidad de las fluctuaciones ($\Gamma$) está determinada por la disipación ($\gamma$) y la temperatura ($T$). Es una manifestación profunda del equilibrio térmico.

### 2.6 Proceso de Wiener

En el límite sobreamortiguado ($m \to 0$), la posición sigue el **proceso de Wiener** $W(t)$:

$$dx = \sqrt{2D} \, dW(t)$$

con propiedades:
1. $W(0) = 0$
2. **Incrementos independientes:** $W(t) - W(s)$ independiente de $W(u)$ para $u \leq s < t$
3. **Incrementos gaussianos:** $W(t) - W(s) \sim \mathcal{N}(0, t-s)$
4. **Continuo pero en ningún punto diferenciable** (casi seguramente)

In [ ]:
# ============================================================
# Simulación de Caminatas Aleatorias y Verificación de ⟨x²⟩ = 2Dt
# ============================================================

np.random.seed(42)

# --- 1. Múltiples trayectorias de caminata aleatoria ---
N_steps = 1000        # Pasos por trayectoria
N_walkers = 500       # Número de caminantes
dt = 0.01             # Paso temporal
D = 1.0               # Coeficiente de difusión

# Incrementos gaussianos: dx ~ N(0, 2D*dt)
sigma = np.sqrt(2 * D * dt)
increments = np.random.normal(0, sigma, size=(N_walkers, N_steps))
positions = np.cumsum(increments, axis=1)
times = np.arange(1, N_steps + 1) * dt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Trayectorias individuales
ax1 = axes[0, 0]
for i in range(20):
    ax1.plot(times, positions[i], alpha=0.5, linewidth=0.7)
ax1.set_xlabel('Tiempo $t$')
ax1.set_ylabel('Posición $x(t)$')
ax1.set_title('Trayectorias de Caminatas Aleatorias (20 de 500)')
ax1.axhline(y=0, color='k', linewidth=0.5, linestyle='--')
ax1.grid(True, alpha=0.3)

# Panel 2: Desplazamiento cuadrático medio ⟨x²⟩
msd = np.mean(positions**2, axis=0)
ax2 = axes[0, 1]
ax2.plot(times, msd, 'b-', linewidth=2, label=r'$\langle x^2 \rangle$ (simulación)')
ax2.plot(times, 2 * D * times, 'r--', linewidth=2, label=r'$2Dt$ (teórico)')
ax2.set_xlabel('Tiempo $t$')
ax2.set_ylabel(r'$\langle x^2 \rangle$')
ax2.set_title('Verificación: Ley de Einstein $\\langle x^2 \\rangle = 2Dt$')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# Panel 3: Evolución del histograma
ax3 = axes[1, 0]
t_indices = [50, 200, 500, 999]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(t_indices)))
for idx, t_idx in enumerate(t_indices):
    t_val = times[t_idx]
    ax3.hist(positions[:, t_idx], bins=40, density=True, alpha=0.4, 
             color=colors[idx], label=f'$t = {t_val:.1f}$')
    # Gaussiana teórica
    x_range = np.linspace(-8, 8, 300)
    gaussian = (1 / np.sqrt(4 * np.pi * D * t_val)) * np.exp(-x_range**2 / (4 * D * t_val))
    ax3.plot(x_range, gaussian, color=colors[idx], linewidth=2)
ax3.set_xlabel('Posición $x$')
ax3.set_ylabel('Densidad de probabilidad')
ax3.set_title('Evolución de $P(x,t)$: Histograma vs Gaussiana teórica')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Panel 4: Verificación de la distribución gaussiana (Q-Q plot)
ax4 = axes[1, 1]
from scipy import stats
final_positions = positions[:, -1]
t_final = times[-1]
theoretical_std = np.sqrt(2 * D * t_final)
(osm, osr), (slope, intercept, r) = stats.probplot(final_positions / theoretical_std, dist="norm")
ax4.plot(osm, osr, 'bo', markersize=3, alpha=0.5)
ax4.plot(osm, slope * np.array(osm) + intercept, 'r-', linewidth=2)
ax4.set_xlabel('Cuantiles teóricos')
ax4.set_ylabel('Cuantiles muestrales')
ax4.set_title(f'Q-Q Plot (normalizado): $R^2 = {r**2:.6f}$')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Estadísticas ---
print("=" * 60)
print("VERIFICACIÓN ESTADÍSTICA")
print("=" * 60)
print(f"⟨x²⟩ final (simulación): {msd[-1]:.4f}")
print(f"2Dt final (teórico):      {2*D*times[-1]:.4f}")
print(f"Error relativo:           {abs(msd[-1] - 2*D*times[-1])/(2*D*times[-1])*100:.2f}%")
print(f"⟨x⟩ final:               {np.mean(positions[:, -1]):.4f} (debe ser ≈ 0)")

## 📐 3. Ecuación de Fokker-Planck

### 3.1 Derivación desde Chapman-Kolmogorov

La **ecuación de Fokker-Planck** (también conocida como ecuación de Kolmogorov hacia adelante) describe la evolución temporal de la densidad de probabilidad $P(x,t)$ de un proceso estocástico.

Partimos de la **ecuación de Chapman-Kolmogorov**:

$$P(x, t+\Delta t) = \int_{-\infty}^{\infty} P(x - \xi, t) \, \phi(\xi; x-\xi, t) \, d\xi$$

donde $\phi(\xi; x', t)$ es la probabilidad de transición de dar un salto $\xi$ desde $x'$ en tiempo $\Delta t$.

**Expansión de Kramers-Moyal:** Expandimos $P(x-\xi, t)\phi(\xi)$ en serie de Taylor alrededor de $\xi = 0$:

$$P(x, t+\Delta t) = \sum_{n=0}^{\infty} \frac{(-1)^n}{n!} \frac{\partial^n}{\partial x^n} \left[ M_n(x,t) \, P(x,t) \right]$$

donde los **momentos de Kramers-Moyal** son:

$$M_n(x,t) = \int_{-\infty}^{\infty} \xi^n \, \phi(\xi; x, t) \, d\xi$$

### 3.2 Truncamiento y Ecuación de Fokker-Planck

Para procesos con saltos pequeños (como el movimiento browniano), truncamos al segundo orden. Definimos:

- **Coeficiente de deriva (drift):** $A(x,t) = \lim_{\Delta t \to 0} \frac{M_1}{\Delta t} = \lim_{\Delta t \to 0} \frac{\langle \Delta x \rangle}{\Delta t}$
- **Coeficiente de difusión:** $B(x,t) = \lim_{\Delta t \to 0} \frac{M_2}{\Delta t} = \lim_{\Delta t \to 0} \frac{\langle (\Delta x)^2 \rangle}{\Delta t}$

Obtenemos la **ecuación de Fokker-Planck**:

$$\boxed{\frac{\partial P}{\partial t} = -\frac{\partial}{\partial x}\left[A(x,t) \, P\right] + \frac{1}{2}\frac{\partial^2}{\partial x^2}\left[B(x,t) \, P\right]}$$

### 3.3 Caso del Movimiento Browniano Libre

Para difusión libre: $A(x,t) = 0$ y $B(x,t) = 2D$, lo que da la **ecuación de difusión**:

$$\frac{\partial P}{\partial t} = D \frac{\partial^2 P}{\partial x^2}$$

cuya solución con condición inicial $P(x,0) = \delta(x)$ es la distribución gaussiana:

$$P(x,t) = \frac{1}{\sqrt{4\pi D t}} \exp\left(-\frac{x^2}{4Dt}\right)$$

### 3.4 Conexión con Langevin

Para la ecuación de Langevin $\dot{x} = -\frac{\gamma}{m}x + \sqrt{2D}\,\eta(t)$, los coeficientes son $A = -\gamma x/m$ y $B = 2D$, dando:

$$\frac{\partial P}{\partial t} = \frac{\gamma}{m}\frac{\partial}{\partial x}(x P) + D \frac{\partial^2 P}{\partial x^2}$$

que en el estado estacionario ($\partial_t P = 0$) da la distribución de **Maxwell-Boltzmann**.

In [ ]:
# ============================================================
# Solución Numérica de la Ecuación de Fokker-Planck
# ============================================================

# --- 1. Método de diferencias finitas (Crank-Nicolson) ---

D_fp = 1.0  # Coeficiente de difusión

# Dominio espacial
L_domain = 15.0
Nx = 500
dx = 2 * L_domain / Nx
x_grid = np.linspace(-L_domain, L_domain, Nx + 1)

# Dominio temporal
T_final = 3.0
Nt = 3000
dt_fp = T_final / Nt
t_grid = np.linspace(0, T_final, Nt + 1)

# Número de Courant
r = D_fp * dt_fp / dx**2

# Condición inicial: delta de Dirac (aproximada por Gaussiana estrecha)
sigma_0 = 0.2
P = np.exp(-x_grid**2 / (2 * sigma_0**2)) / (sigma_0 * np.sqrt(2 * np.pi))

# Matrices para Crank-Nicolson: (I + r/2 A)P^{n+1} = (I - r/2 A)P^n
# donde A es la matriz tridiagonal del Laplaciano

# Construir matrices tridiagonales
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# Diagonal principal y secundarias
main_diag_lhs = np.ones(Nx + 1) * (1 + r)
off_diag_lhs = np.ones(Nx) * (-r / 2)
main_diag_rhs = np.ones(Nx + 1) * (1 - r)
off_diag_rhs = np.ones(Nx) * (r / 2)

# Condiciones de frontera: P(-L) = P(L) = 0
main_diag_lhs[0] = main_diag_lhs[-1] = 1
off_diag_lhs[0] = off_diag_lhs[-1] = 0
main_diag_rhs[0] = main_diag_rhs[-1] = 0
off_diag_rhs[0] = off_diag_rhs[-1] = 0

A_lhs = diags([off_diag_lhs, main_diag_lhs, off_diag_lhs], [-1, 0, 1], format='csc')
A_rhs = diags([off_diag_rhs, main_diag_rhs, off_diag_rhs], [-1, 0, 1], format='csc')

# Almacenar soluciones en tiempos seleccionados
snapshot_times = [0.0, 0.1, 0.3, 0.5, 1.0, 2.0, 3.0]
snapshots = {0.0: P.copy()}

# Evolución temporal
for n in range(Nt):
    rhs = A_rhs @ P
    P = spsolve(A_lhs, rhs)
    P[0] = P[-1] = 0  # Frontera absorbente
    
    current_t = (n + 1) * dt_fp
    for st in snapshot_times:
        if abs(current_t - st) < dt_fp / 2 and st not in snapshots:
            snapshots[st] = P.copy()

# --- Visualización ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Evolución temporal de P(x,t)
ax1 = axes[0, 0]
colors_fp = plt.cm.plasma(np.linspace(0, 0.9, len(snapshot_times)))

for (t_snap, P_snap), color in zip(sorted(snapshots.items()), colors_fp):
    ax1.plot(x_grid, P_snap, color=color, linewidth=2, label=f'$t = {t_snap}$')
    
    # Solución analítica gaussiana
    if t_snap > 0:
        sigma_t = np.sqrt(sigma_0**2 + 2 * D_fp * t_snap)
        P_analytical = np.exp(-x_grid**2 / (2 * sigma_t**2)) / (sigma_t * np.sqrt(2 * np.pi))
        ax1.plot(x_grid, P_analytical, '--', color=color, alpha=0.5, linewidth=1.5)

ax1.set_xlabel('Posición $x$')
ax1.set_ylabel('$P(x, t)$')
ax1.set_title('Ecuación de Fokker-Planck: Solución Numérica (—) vs Analítica (--)')
ax1.legend(fontsize=9, loc='upper right')
ax1.set_xlim(-8, 8)
ax1.grid(True, alpha=0.3)

# Panel 2: Error numérico vs analítico
ax2 = axes[0, 1]

errors = {}
for t_snap, P_snap in sorted(snapshots.items()):
    if t_snap > 0:
        sigma_t = np.sqrt(sigma_0**2 + 2 * D_fp * t_snap)
        P_analytical = np.exp(-x_grid**2 / (2 * sigma_t**2)) / (sigma_t * np.sqrt(2 * np.pi))
        error = np.abs(P_snap - P_analytical)
        errors[t_snap] = np.max(error)
        if t_snap in [0.1, 0.5, 1.0, 3.0]:
            ax2.plot(x_grid, error, linewidth=1.5, label=f'$t = {t_snap}$')

ax2.set_xlabel('Posición $x$')
ax2.set_ylabel('Error $|P_{\\rm num} - P_{\\rm anal}|$')
ax2.set_title('Error del Método de Crank-Nicolson')
ax2.legend(fontsize=10)
ax2.set_xlim(-8, 8)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# Panel 3: Verificación de conservación de probabilidad y momentos
ax3 = axes[1, 0]

# Re-ejecutar para recopilar estadísticas
P_check = np.exp(-x_grid**2 / (2 * sigma_0**2)) / (sigma_0 * np.sqrt(2 * np.pi))
norms = [np.trapz(P_check, x_grid)]
means = [np.trapz(x_grid * P_check, x_grid)]
variances = [np.trapz(x_grid**2 * P_check, x_grid)]
t_record = [0.0]

for n in range(Nt):
    rhs = A_rhs @ P_check
    P_check = spsolve(A_lhs, rhs)
    P_check[0] = P_check[-1] = 0
    
    if (n + 1) % 50 == 0:
        norm = np.trapz(P_check, x_grid)
        mean = np.trapz(x_grid * P_check, x_grid)
        var = np.trapz(x_grid**2 * P_check, x_grid)
        norms.append(norm)
        means.append(mean)
        variances.append(var)
        t_record.append((n + 1) * dt_fp)

t_record = np.array(t_record)
variances = np.array(variances)

# Varianza teórica: σ²(t) = σ₀² + 2Dt
var_theory = sigma_0**2 + 2 * D_fp * t_record

ax3.plot(t_record, variances, 'b-', linewidth=2, label=r'$\langle x^2 \rangle$ numérico')
ax3.plot(t_record, var_theory, 'r--', linewidth=2, label=r'$\sigma_0^2 + 2Dt$ (teórico)')
ax3.set_xlabel('Tiempo $t$')
ax3.set_ylabel(r'$\langle x^2(t) \rangle$')
ax3.set_title(r'Verificación: $\langle x^2 \rangle = \sigma_0^2 + 2Dt$')
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)

# Panel 4: Mapa de calor espacio-tiempo
ax4 = axes[1, 1]

# Recalcular con almacenamiento completo (submuestreado)
P_heat = np.exp(-x_grid**2 / (2 * sigma_0**2)) / (sigma_0 * np.sqrt(2 * np.pi))
save_every = 20
P_matrix = [P_heat.copy()]
t_heat = [0.0]

for n in range(Nt):
    rhs = A_rhs @ P_heat
    P_heat = spsolve(A_lhs, rhs)
    P_heat[0] = P_heat[-1] = 0
    if (n + 1) % save_every == 0:
        P_matrix.append(P_heat.copy())
        t_heat.append((n + 1) * dt_fp)

P_matrix = np.array(P_matrix)
t_heat = np.array(t_heat)

# Recortar dominio para mejor visualización
x_mask = (x_grid > -6) & (x_grid < 6)

im = ax4.pcolormesh(x_grid[x_mask], t_heat, P_matrix[:, x_mask], 
                     cmap='hot_r', shading='auto')
plt.colorbar(im, ax=ax4, label='$P(x, t)$')

# Superponer contornos de ±σ(t)
sigma_envelope = np.sqrt(sigma_0**2 + 2 * D_fp * t_heat)
ax4.plot(sigma_envelope, t_heat, 'c--', linewidth=2, label=r'$\pm\sigma(t) = \pm\sqrt{2Dt}$')
ax4.plot(-sigma_envelope, t_heat, 'c--', linewidth=2)

ax4.set_xlabel('Posición $x$')
ax4.set_ylabel('Tiempo $t$')
ax4.set_title('Diagrama Espacio-Tiempo de $P(x, t)$')
ax4.legend(fontsize=10, loc='upper right')

plt.suptitle('Ecuación de Fokker-Planck: Difusión Libre', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Resumen numérico
print("=" * 60)
print("VERIFICACIÓN DE LA SOLUCIÓN FOKKER-PLANCK")
print("=" * 60)
print(f"Método: Crank-Nicolson (incondicionalmente estable)")
print(f"Nx = {Nx}, dx = {dx:.4f}, Nt = {Nt}, dt = {dt_fp:.5f}")
print(f"Número de Courant: r = D·dt/dx² = {r:.4f}")
print(f"Conservación de norma: {norms[-1]:.6f} (debe ser ≈ 1)")
print(f"Error máximo respecto a solución analítica:")
for t_e, err in sorted(errors.items()):
    print(f"  t = {t_e:.1f}: max|error| = {err:.2e}")

## 🎯 4. Tiempos de Primer Paso y Problemas de Frontera

### 4.1 Definición del Tiempo de Primer Paso

El **tiempo de primer paso** (FPT, *first passage time*) es un concepto fundamental en la teoría de procesos estocásticos con aplicaciones en física, química, biología y finanzas.

**Definición**: Dado un proceso estocástico $X(t)$ con condición inicial $X(0) = x_0$, el tiempo de primer paso a un nivel $L$ es:

$$T_L = \inf\{t > 0 : X(t) = L\}$$

### 4.2 Distribución del FPT para Difusión Libre

Para un movimiento browniano $X(t)$ con $X(0) = 0$ y coeficiente de difusión $D$, la distribución del primer paso al nivel $L > 0$ es la **distribución de Lévy inversa** (o distribución de Wald):

$$\boxed{f(t; L) = \frac{L}{\sqrt{4\pi D\, t^3}} \, \exp\!\left(-\frac{L^2}{4Dt}\right), \quad t > 0}$$

**Propiedades notables:**
- La distribución tiene **cola pesada**: $f(t) \sim t^{-3/2}$ para $t \to \infty$
- El **momento medio es infinito**: $\langle T_L \rangle = \infty$ (para difusión en semilínea)
- La **mediana** es finita y escala como $\text{Med}(T) \sim L^2 / D$

### 4.3 Derivación por el Método de las Imágenes

La distribución del FPT se obtiene del **método de las imágenes** aplicado a la ecuación de difusión con condición de frontera absorbente $P(L, t) = 0$:

$$P(x, t | 0, 0) = \frac{1}{\sqrt{4\pi Dt}}\left[e^{-x^2/(4Dt)} - e^{-(x-2L)^2/(4Dt)}\right]$$

La densidad del FPT es entonces el **flujo de probabilidad** en $x = L$:

$$f(t) = -\frac{\partial}{\partial t}\int_0^L P(x, t)\,dx = D\,\frac{\partial P}{\partial x}\bigg|_{x=L}$$

### 4.4 Probabilidad de Supervivencia

La **probabilidad de supervivencia** $S(t) = P(T > t)$ es la probabilidad de que la partícula no haya alcanzado la barrera hasta el tiempo $t$:

$$S(t) = \int_0^L P(x, t)\,dx = \text{erf}\!\left(\frac{L}{\sqrt{4Dt}}\right)$$

Para tiempos largos: $S(t) \sim \frac{L}{\sqrt{\pi D t}} \to 0$, confirmando que la partícula **siempre** alcanza la barrera eventualmente.

### 4.5 MFPT en Dominios Acotados

En un intervalo $[0, L]$ con **condiciones de frontera absorbentes** en ambos extremos, el tiempo medio de primer paso desde $x_0$ satisface la **ecuación de Dynkin**:

$$D\,\frac{d^2 \langle T \rangle}{dx_0^2} = -1, \quad \langle T \rangle(0) = \langle T \rangle(L) = 0$$

La solución es:

$$\boxed{\langle T(x_0) \rangle = \frac{x_0(L - x_0)}{2D}}$$

El máximo se alcanza en el centro $x_0 = L/2$: $\langle T_{\rm max} \rangle = L^2 / (8D)$.

### 4.6 Aplicaciones del FPT

| Área | Problema | Variable $X(t)$ |
|------|----------|-----------------|
| **Química** | Tiempo de reacción (Kramers) | Coordenada de reacción |
| **Neurociencia** | Disparo neuronal (integrate-and-fire) | Potencial de membrana |
| **Finanzas** | Tiempo hasta default | Valor de activos |
| **Biología** | Tiempo de búsqueda de una proteína | Posición en ADN |

In [ ]:
# ============================================================
# Simulación de Tiempos de Primer Paso
# ============================================================

np.random.seed(42)

# --- 1. Tiempo de primer paso a una barrera absorbente ---
D_fpt = 1.0          # Coeficiente de difusión
L_barrier = 5.0      # Posición de la barrera
x0_values = [0.0, 1.0, 2.5]  # Posiciones iniciales
dt = 0.001
max_steps = 200000
n_trials = 5000

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Trayectorias ejemplo con barrera
ax1 = axes[0, 0]
ax1.axhline(y=L_barrier, color='red', linewidth=2, linestyle='--', label=f'Barrera $L = {L_barrier}$')
ax1.axhline(y=-L_barrier, color='red', linewidth=2, linestyle='--')

colors_traj = plt.cm.viridis(np.linspace(0.2, 0.8, 10))
for j in range(10):
    x = 0.0
    trajectory = [x]
    for _ in range(50000):
        x += np.sqrt(2 * D_fpt * dt) * np.random.randn()
        trajectory.append(x)
        if abs(x) >= L_barrier:
            break
    ax1.plot(np.arange(len(trajectory)) * dt, trajectory, 
             color=colors_traj[j], alpha=0.6, linewidth=0.8)

ax1.set_xlabel('Tiempo $t$')
ax1.set_ylabel('Posición $x(t)$')
ax1.set_title('Trayectorias con Barreras Absorbentes')
ax1.set_ylim(-L_barrier - 1, L_barrier + 1)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Panel 2: Distribución de tiempos de primer paso (barrera en +L, partiendo de x=0)
ax2 = axes[0, 1]

fpt_times = []
for trial in range(n_trials):
    x = 0.0
    for step in range(max_steps):
        x += np.sqrt(2 * D_fpt * dt) * np.random.randn()
        if x >= L_barrier:
            fpt_times.append((step + 1) * dt)
            break

fpt_times = np.array(fpt_times)

# Distribución teórica del FPT para difusión libre a barrera en L desde x=0:
# f(t) = L / sqrt(4πDt³) * exp(-L²/(4Dt))  (distribución de Lévy inversa)
t_theory = np.linspace(0.1, np.percentile(fpt_times, 95), 500)
f_fpt_theory = L_barrier / np.sqrt(4 * np.pi * D_fpt * t_theory**3) * \
               np.exp(-L_barrier**2 / (4 * D_fpt * t_theory))

ax2.hist(fpt_times, bins=80, density=True, alpha=0.6, color='steelblue',
         label=f'Simulación ($N = {len(fpt_times)}$)')
ax2.plot(t_theory, f_fpt_theory, 'r-', linewidth=2.5, 
         label=r'$f(t) = \frac{L}{\sqrt{4\pi D t^3}} e^{-L^2/(4Dt)}$')
ax2.set_xlabel('Tiempo de primer paso $T$')
ax2.set_ylabel('Densidad de probabilidad $f(T)$')
ax2.set_title(f'Distribución de FPT ($x_0 = 0$, barrera en $L = {L_barrier}$)')
ax2.legend(fontsize=9)
ax2.set_xlim(0, np.percentile(fpt_times, 95))
ax2.grid(True, alpha=0.3)

# Panel 3: Probabilidad de supervivencia S(t)
ax3 = axes[1, 0]

# S(t) = P(T > t) = 1 - CDF(t)
t_sorted = np.sort(fpt_times)
survival_sim = 1 - np.arange(1, len(t_sorted) + 1) / len(t_sorted)

# Teórica: S(t) = erf(L / sqrt(4Dt))
from scipy.special import erf, erfc
t_surv = np.linspace(0.01, t_sorted[-1], 1000)
survival_theory = erf(L_barrier / np.sqrt(4 * D_fpt * t_surv))

ax3.plot(t_sorted, survival_sim, 'b-', linewidth=1.5, label='Simulación')
ax3.plot(t_surv, survival_theory, 'r--', linewidth=2, 
         label=r'$S(t) = \mathrm{erf}\!\left(\frac{L}{\sqrt{4Dt}}\right)$')
ax3.set_xlabel('Tiempo $t$')
ax3.set_ylabel('Probabilidad de supervivencia $S(t)$')
ax3.set_title('Probabilidad de Supervivencia')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Panel 4: MFPT como función de la posición inicial (intervalo [0, L])
ax4 = axes[1, 1]

# Para difusión en intervalo [0, L] con absorbente en 0 y L:
# MFPT(x₀) = x₀(L - x₀) / (2D)

L_interval = 5.0
x0_range = np.linspace(0.1, L_interval - 0.1, 20)
mfpt_sim = np.zeros(len(x0_range))
mfpt_theory_range = x0_range * (L_interval - x0_range) / (2 * D_fpt)

n_trials_mfpt = 2000
for idx, x0 in enumerate(x0_range):
    times = []
    for _ in range(n_trials_mfpt):
        x = x0
        for step in range(max_steps):
            x += np.sqrt(2 * D_fpt * dt) * np.random.randn()
            if x <= 0 or x >= L_interval:
                times.append((step + 1) * dt)
                break
    mfpt_sim[idx] = np.mean(times) if times else np.nan

ax4.plot(x0_range, mfpt_sim, 'bo-', markersize=6, linewidth=1.5, label='Simulación')
ax4.plot(x0_range, mfpt_theory_range, 'r-', linewidth=2.5, 
         label=r'$\langle T \rangle = \frac{x_0(L-x_0)}{2D}$')
ax4.set_xlabel('Posición inicial $x_0$')
ax4.set_ylabel(r'MFPT $\langle T \rangle$')
ax4.set_title(f'Tiempo Medio de Primer Paso en $[0, {L_interval}]$')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

plt.suptitle('Tiempos de Primer Paso del Movimiento Browniano', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Resultados numéricos
print("=" * 60)
print("RESULTADOS DE TIEMPOS DE PRIMER PASO")
print("=" * 60)
print(f"Barrera en L = {L_barrier}, D = {D_fpt}")
print(f"Partículas que alcanzaron la barrera: {len(fpt_times)}/{n_trials} ({100*len(fpt_times)/n_trials:.1f}%)")
print(f"⟨T⟩ simulado = {np.mean(fpt_times):.3f}")
print(f"⟨T⟩ teórico (semi-infinito) = ∞ (la distribución de Lévy inversa tiene media infinita)")
print(f"Mediana(T) simulada = {np.median(fpt_times):.3f}")
print(f"\nMFPT en intervalo [0, {L_interval}] desde x₀ = L/2:")
print(f"  Simulado: {mfpt_sim[len(x0_range)//2]:.3f}")
print(f"  Teórico:  {L_interval**2 / 8:.3f}")

## ⚡ 5. Proceso de Ornstein-Uhlenbeck

### 5.1 Motivación: La Velocidad de una Partícula Browniana

El proceso de Wiener modela la **posición** de una partícula browniana en el límite sobreaamortiguado ($m \to 0$). Sin embargo, para describir la **velocidad** $v(t)$, necesitamos un proceso que:

1. Satisfaga el **teorema de equipartición**: $\frac{1}{2}m\langle v^2 \rangle = \frac{1}{2}k_B T$
2. Tenga una **distribución estacionaria** de Maxwell-Boltzmann
3. Muestre **reversión a la media**: $v(t) \to 0$ en promedio

El **proceso de Ornstein-Uhlenbeck** (OU) satisface todas estas condiciones.

### 5.2 Ecuación de Langevin para la Velocidad

Partiendo de la ecuación de Langevin $m\dot{v} = -\gamma v + \eta(t)$, dividiendo por $m$:

$$\boxed{dv(t) = -\theta \, v(t)\,dt + \sigma\, dW(t)}$$

donde:
- $\theta = \gamma/m$ es la **tasa de relajación** (inverso del tiempo de correlación)
- $\sigma = \sqrt{2\gamma k_B T / m^2}$ es la **amplitud del ruido**
- $dW(t)$ es el incremento del proceso de Wiener

### 5.3 Solución Exacta

La ecuación OU es una **EDE lineal** con solución explícita (usando factor integrante $e^{\theta t}$):

$$v(t) = v_0 \, e^{-\theta t} + \sigma \int_0^t e^{-\theta(t-s)} \, dW(s)$$

**Momentos:**

**Valor esperado** (decaimiento exponencial hacia el equilibrio):
$$\langle v(t) \rangle = v_0 \, e^{-\theta t}$$

**Varianza** (crecimiento hacia el valor de equipartición):
$$\text{Var}\big(v(t)\big) = \frac{\sigma^2}{2\theta}\left(1 - e^{-2\theta t}\right) \xrightarrow{t \to \infty} \frac{\sigma^2}{2\theta} = \frac{k_B T}{m}$$

### 5.4 Función de Autocorrelación

La función de autocorrelación de velocidades en el **estado estacionario** es:

$$C_v(\tau) = \langle v(t+\tau)\,v(t) \rangle_{\rm est} = \frac{k_B T}{m} \, e^{-\theta|\tau|}$$

Esta es una propiedad central que conecta con el **teorema de Green-Kubo**:

$$D = \int_0^\infty C_v(\tau)\,d\tau = \frac{k_B T}{m} \cdot \frac{1}{\theta} = \frac{k_B T}{\gamma}$$

recuperando la **relación de Einstein** $D = k_B T / \gamma$.

### 5.5 MSD del Proceso OU Completo

Integrando la velocidad para obtener la posición $x(t) = \int_0^t v(s)\,ds$:

$$\langle x^2(t) \rangle = \frac{2k_B T}{\gamma}\left[t - \frac{m}{\gamma}\left(1 - e^{-\gamma t/m}\right)\right]$$

Esto exhibe dos **regímenes**:

- **Tiempos cortos** ($t \ll m/\gamma$): $\langle x^2 \rangle \approx \frac{k_B T}{m} t^2$ (régimen **balístico**, $\alpha = 2$)
- **Tiempos largos** ($t \gg m/\gamma$): $\langle x^2 \rangle \approx 2Dt$ (régimen **difusivo**, $\alpha = 1$)

El **tiempo de cruce** $\tau_c = m/\gamma$ separa ambos regímenes. Para una partícula coloidal típica en agua, $\tau_c \sim 10^{-8}$ s.

### 5.6 Distribución Estacionaria

En el estado estacionario ($t \to \infty$), la distribución de velocidades converge a la **distribución de Maxwell-Boltzmann**:

$$P_{\rm est}(v) = \sqrt{\frac{m}{2\pi k_B T}} \, \exp\!\left(-\frac{mv^2}{2k_B T}\right)$$

El proceso OU es **ergódico**: los promedios temporales coinciden con los promedios de ensamble.

In [ ]:
# ============================================================
# Simulación del Proceso de Ornstein-Uhlenbeck
# ============================================================

np.random.seed(42)

# Parámetros físicos
gamma_ou = 5.0      # Coeficiente de fricción γ
m_ou = 1.0          # Masa
kBT_ou = 1.0        # Energía térmica k_B T
theta = gamma_ou / m_ou  # Tasa de relajación θ = γ/m
sigma_ou = np.sqrt(2 * gamma_ou * kBT_ou / m_ou**2)  # Intensidad del ruido

# Parámetros de simulación
dt = 0.005
T_total = 10.0
n_steps = int(T_total / dt)
n_particles = 500
t_array = np.arange(n_steps + 1) * dt

# --- 1. Simulación del proceso OU para velocidad ---
# dv = -θ v dt + σ dW

v_ensemble = np.zeros((n_particles, n_steps + 1))
v_ensemble[:, 0] = 3.0  # Condición inicial: v₀ = 3.0 (fuera del equilibrio)

for i in range(n_steps):
    dW = np.sqrt(dt) * np.random.randn(n_particles)
    v_ensemble[:, i+1] = v_ensemble[:, i] - theta * v_ensemble[:, i] * dt + sigma_ou * dW

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Panel 1: Trayectorias de velocidad
ax1 = axes[0, 0]
for j in range(15):
    ax1.plot(t_array, v_ensemble[j], alpha=0.4, linewidth=0.7)

# Media teórica: ⟨v(t)⟩ = v₀ exp(-θt)
v_mean_theory = 3.0 * np.exp(-theta * t_array)
v_mean_sim = np.mean(v_ensemble, axis=0)

ax1.plot(t_array, v_mean_theory, 'r-', linewidth=2.5, label=r'$\langle v \rangle_{\rm teórico} = v_0 e^{-\theta t}$')
ax1.plot(t_array, v_mean_sim, 'k--', linewidth=2, label=r'$\langle v \rangle_{\rm simulado}$')
ax1.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Tiempo $t$')
ax1.set_ylabel('Velocidad $v(t)$')
ax1.set_title('Proceso Ornstein-Uhlenbeck: Velocidad')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Panel 2: Varianza de la velocidad
ax2 = axes[0, 1]
v_var_sim = np.var(v_ensemble, axis=0)
# Varianza teórica: Var(v(t)) = (σ²/2θ)(1 - exp(-2θt))
v_var_theory = (sigma_ou**2 / (2 * theta)) * (1 - np.exp(-2 * theta * t_array))
v_var_equil = kBT_ou / m_ou  # Equipartición: (1/2)m⟨v²⟩ = (1/2)k_BT

ax2.plot(t_array, v_var_sim, 'b-', linewidth=2, label='Simulación')
ax2.plot(t_array, v_var_theory, 'r--', linewidth=2, label=r'$\frac{\sigma^2}{2\theta}(1-e^{-2\theta t})$')
ax2.axhline(y=v_var_equil, color='green', linestyle=':', linewidth=1.5, 
            label=r'Equipartición: $k_BT/m$')
ax2.set_xlabel('Tiempo $t$')
ax2.set_ylabel(r'Var$(v)$')
ax2.set_title('Varianza de la Velocidad')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Panel 3: Autocorrelación de velocidad
ax3 = axes[0, 2]

# Calcular autocorrelación del ensemble en estado estacionario
# Usamos la segunda mitad de la simulación (estado estacionario)
start_idx = n_steps // 2
v_steady = v_ensemble[:, start_idx:]
n_steady = v_steady.shape[1]

max_lag = min(400, n_steady // 2)
lags = np.arange(max_lag)
tau_lags = lags * dt

autocorr_sim = np.zeros(max_lag)
for lag in range(max_lag):
    if lag == 0:
        autocorr_sim[0] = np.mean(v_steady[:, :n_steady]**2)
    else:
        autocorr_sim[lag] = np.mean(v_steady[:, :n_steady-lag] * v_steady[:, lag:n_steady])

autocorr_sim /= autocorr_sim[0]  # Normalizar

# Teórico: C(τ) = exp(-θ|τ|)
autocorr_theory = np.exp(-theta * tau_lags)

ax3.plot(tau_lags, autocorr_sim, 'b-', linewidth=2, label='Simulación')
ax3.plot(tau_lags, autocorr_theory, 'r--', linewidth=2, label=r'$e^{-\theta|\tau|}$')
ax3.set_xlabel(r'Retardo $\tau$')
ax3.set_ylabel(r'$C(\tau) / C(0)$')
ax3.set_title('Autocorrelación de Velocidad')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# --- 2. Posición: integrar v para obtener x ---
x_ensemble = np.cumsum(v_ensemble * dt, axis=1)

# Panel 4: Trayectorias de posición
ax4 = axes[1, 0]
for j in range(15):
    ax4.plot(t_array, x_ensemble[j], alpha=0.4, linewidth=0.7)
ax4.plot(t_array, np.mean(x_ensemble, axis=0), 'k-', linewidth=2.5, label=r'$\langle x(t) \rangle$')
ax4.set_xlabel('Tiempo $t$')
ax4.set_ylabel('Posición $x(t)$')
ax4.set_title('Posición de Partículas OU')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

# Panel 5: MSD de la posición
ax5 = axes[1, 1]
msd_ou = np.mean(x_ensemble**2, axis=0) - np.mean(x_ensemble, axis=0)**2

# MSD teórico para proceso OU (partiendo de v₀):
# ⟨(Δx)²⟩ = (2k_BT/γ)[t - (m/γ)(1 - e^{-γt/m})]  (régimen difusivo a largo plazo)
# Régimen balístico corto: ⟨x²⟩ ~ v₀²t² para t << m/γ
# Régimen difusivo largo: ⟨x²⟩ ~ 2Dt = 2(k_BT/γ)t para t >> m/γ
D_ou = kBT_ou / gamma_ou
msd_theory_ou = 2 * D_ou * (t_array - (m_ou/gamma_ou)*(1 - np.exp(-gamma_ou*t_array/m_ou)))

ax5.plot(t_array[1:], msd_ou[1:], 'b-', linewidth=2, label='Simulación')
ax5.plot(t_array[1:], msd_theory_ou[1:], 'r--', linewidth=2, label='Teórico (OU)')
ax5.plot(t_array[1:], 2*D_ou*t_array[1:], 'g:', linewidth=1.5, label=r'$2Dt$ (Einstein)')
ax5.set_xlabel('Tiempo $t$')
ax5.set_ylabel(r'$\langle (\Delta x)^2 \rangle$')
ax5.set_title('MSD: Transición Balístico → Difusivo')
ax5.legend(fontsize=9)
ax5.set_xscale('log')
ax5.set_yscale('log')
ax5.grid(True, alpha=0.3, which='both')

# Panel 6: Distribución estacionaria de velocidades
ax6 = axes[1, 2]
v_final = v_ensemble[:, -1]
v_range = np.linspace(-4, 4, 200)
# Maxwell-Boltzmann 1D: P(v) = sqrt(m/(2πk_BT)) exp(-mv²/(2k_BT))
maxwell = np.sqrt(m_ou / (2*np.pi*kBT_ou)) * np.exp(-m_ou*v_range**2 / (2*kBT_ou))

ax6.hist(v_final, bins=40, density=True, alpha=0.6, color='steelblue', 
         label='Simulación (estado estacionario)')
ax6.plot(v_range, maxwell, 'r-', linewidth=2.5, 
         label=r'Maxwell-Boltzmann: $\sqrt{\frac{m}{2\pi k_BT}}e^{-\frac{mv^2}{2k_BT}}$')
ax6.set_xlabel('Velocidad $v$')
ax6.set_ylabel('Densidad de probabilidad')
ax6.set_title('Distribución Estacionaria de Velocidades')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3)

plt.suptitle(f'Proceso de Ornstein-Uhlenbeck (θ = {theta}, σ = {sigma_ou:.2f}, v₀ = 3.0)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Verificación numérica
print("=" * 60)
print("VERIFICACIÓN DEL PROCESO ORNSTEIN-UHLENBECK")
print("=" * 60)
print(f"Tiempo de relajación: τ_rel = m/γ = {m_ou/gamma_ou:.3f}")
print(f"⟨v⟩ final (simulación): {np.mean(v_final):.4f} (teórico: 0)")
print(f"Var(v) final (simulación): {np.var(v_final):.4f} (teórico k_BT/m = {kBT_ou/m_ou:.4f})")
print(f"D = k_BT/γ = {D_ou:.4f}")
print(f"Teorema de equipartición: ½m⟨v²⟩ = {0.5*m_ou*np.mean(v_final**2):.4f} (teórico: ½k_BT = {0.5*kBT_ou:.4f})")

## 🌊 6. Difusión Anómala y Extensiones

### 6.1 Más Allá de Einstein: Cuando $\langle x^2 \rangle \neq 2Dt$

En muchos sistemas físicos y biológicos, el desplazamiento cuadrático medio **no** crece linealmente con el tiempo. Se observa en cambio un comportamiento de **ley de potencia**:

$$\boxed{\langle x^2(t) \rangle \sim K_\alpha \, t^\alpha}$$

donde $\alpha$ es el **exponente de difusión anómala**:

| Régimen | Exponente | Mecanismo |
|---------|-----------|-----------|
| **Subdifusión** | $0 < \alpha < 1$ | Trampas, obstáculos, viscoelasticidad |
| **Difusión normal** | $\alpha = 1$ | Movimiento browniano clásico |
| **Superdifusión** | $1 < \alpha < 2$ | Correlaciones persistentes, transporte activo |
| **Difusión balística** | $\alpha = 2$ | Movimiento libre sin colisiones |

### 6.2 Movimiento Browniano Fraccional (fBm)

El **movimiento browniano fraccional** $B_H(t)$ generaliza el proceso de Wiener introduciendo correlaciones a largo plazo, parametrizadas por el **exponente de Hurst** $H \in (0, 1)$:

**Definición** (Mandelbrot & Van Ness, 1968): $B_H(t)$ es el proceso gaussiano centrado con función de covarianza:

$$\text{Cov}\big(B_H(s),\, B_H(t)\big) = \frac{1}{2}\left(|s|^{2H} + |t|^{2H} - |t-s|^{2H}\right)$$

**Propiedades fundamentales:**

1. **Autosimilaridad**: $B_H(ct) \stackrel{d}{=} c^H B_H(t)$ para todo $c > 0$

2. **Incrementos estacionarios** pero **no independientes** (para $H \neq 1/2$):
$$\text{Corr}\big(\Delta B_H(t),\, \Delta B_H(t+\tau)\big) \sim H(2H-1)|\tau|^{2H-2}$$
   - $H < 1/2$: correlaciones **negativas** (antipersistencia) $\Rightarrow$ subdifusión
   - $H = 1/2$: movimiento browniano estándar
   - $H > 1/2$: correlaciones **positivas** (persistencia) $\Rightarrow$ superdifusión

3. **MSD**: $\langle B_H^2(t) \rangle = t^{2H}$, por lo que $\alpha = 2H$

4. **Regularidad de Hölder**: las trayectorias son Hölder-continuas de exponente $H - \varepsilon$ para todo $\varepsilon > 0$

### 6.3 Caminatas Aleatorias de Tiempo Continuo (CTRW)

El modelo **CTRW** (Montroll & Weiss, 1965) generaliza la caminata aleatoria permitiendo tiempos de espera aleatorios entre saltos. Si la distribución de tiempos de espera tiene **cola pesada**:

$$\psi(\tau) \sim \frac{A}{\tau^{1+\alpha}}, \quad 0 < \alpha < 1$$

entonces el segundo momento del tiempo de espera **diverge**: $\langle \tau^2 \rangle = \infty$. Esto conduce a **subdifusión** con:

$$\langle x^2(t) \rangle \sim \frac{2K_\alpha}{\Gamma(1+\alpha)} t^\alpha$$

La ecuación de transporte resultante es la **ecuación de difusión fraccional**:

$$\frac{\partial P}{\partial t} = K_\alpha \, {}_0D_t^{1-\alpha} \frac{\partial^2 P}{\partial x^2}$$

donde ${}_0D_t^{1-\alpha}$ es la **derivada fraccional de Riemann-Liouville**.

### 6.4 Vuelos de Lévy

Los **vuelos de Lévy** surgen cuando la distribución de longitudes de salto tiene colas pesadas:

$$P(\ell) \sim \frac{A}{|\ell|^{1+\mu}}, \quad 0 < \mu < 2$$

Para $\mu < 2$, la varianza del salto diverge: $\langle \ell^2 \rangle = \infty$. El **teorema del límite central generalizado** da lugar a **distribuciones estables de Lévy** $L_\mu(x)$ en vez de gaussianas.

**Características de los vuelos de Lévy:**
- Saltos ocasionales extraordinariamente largos
- Estrategia de búsqueda óptima para recursos escasos (hipótesis de Lévy en ecología)
- La distribución de Cauchy ($\mu = 1$) y la gaussiana ($\mu = 2$) son casos particulares

### 6.5 Ecuación de Langevin Generalizada (GLE)

Para sistemas con **memoria**, la ecuación de Langevin se generaliza:

$$m\ddot{x}(t) = -\int_0^t \gamma(t-s)\,\dot{x}(s)\,ds + \eta(t)$$

donde $\gamma(t-s)$ es el **kernel de memoria** (fricción no-markoviana). El teorema de fluctuación-disipación generalizado exige:

$$\langle \eta(t)\eta(s) \rangle = k_B T \, \gamma(|t-s|)$$

Para kernels de ley de potencia $\gamma(t) \sim t^{-\alpha}$, se obtiene subdifusión con $\langle x^2 \rangle \sim t^\alpha$.

In [ ]:
# ============================================================
# Simulación de Difusión Anómala: Subdifusión vs Superdifusión
# ============================================================

np.random.seed(42)

# --- 1. Movimiento Browniano Fraccional (fBm) aproximado ---
# Usamos el método de Cholesky para generar fBm con parámetro de Hurst H

def generate_fbm(n_steps, H, dt=0.01):
    """
    Genera movimiento browniano fraccional usando el método de covarianza.
    H < 0.5: subdifusión (antipersistente)
    H = 0.5: difusión normal (Browniano estándar)
    H > 0.5: superdifusión (persistente)
    """
    # Matriz de covarianza del fBm: Cov(B_H(s), B_H(t)) = 0.5*(|s|^{2H} + |t|^{2H} - |t-s|^{2H})
    times = np.arange(1, n_steps + 1) * dt
    n = len(times)
    
    # Construir matriz de covarianza
    cov = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ti, tj = times[i], times[j]
            cov[i, j] = 0.5 * (ti**(2*H) + tj**(2*H) - abs(ti - tj)**(2*H))
    
    # Regularización para estabilidad numérica
    cov += 1e-10 * np.eye(n)
    
    # Descomposición de Cholesky
    L = np.linalg.cholesky(cov)
    
    # Generar fBm
    z = np.random.randn(n)
    fbm = L @ z
    
    return times, fbm

# Parámetros
n_steps = 300  # Reducido para eficiencia de Cholesky
n_particles = 50
dt = 0.01

H_values = [0.25, 0.5, 0.75]  # Subdifusión, normal, superdifusión
labels = [f'H = {H} ({"subdifusión" if H < 0.5 else "normal" if H == 0.5 else "superdifusión"})' 
          for H in H_values]
colors_H = ['#e74c3c', '#2ecc71', '#3498db']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Trayectorias individuales para cada H
ax1 = axes[0, 0]
for H, label, color in zip(H_values, labels, colors_H):
    times, fbm = generate_fbm(n_steps, H, dt)
    ax1.plot(times, fbm, color=color, label=label, alpha=0.8, linewidth=1.2)

ax1.set_xlabel('Tiempo $t$')
ax1.set_ylabel('Posición $x(t)$')
ax1.set_title('Trayectorias del Movimiento Browniano Fraccional')
ax1.legend(fontsize=10)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax1.grid(True, alpha=0.3)

# Panel 2: MSD para diferentes H (ensemble)
ax2 = axes[0, 1]

for H, label, color in zip(H_values, labels, colors_H):
    msd_ensemble = np.zeros(n_steps)
    for _ in range(n_particles):
        times, fbm = generate_fbm(n_steps, H, dt)
        msd_ensemble += fbm**2
    msd_ensemble /= n_particles
    
    ax2.loglog(times, msd_ensemble, color=color, label=label, linewidth=2)

# Líneas teóricas de referencia
t_ref = times
ax2.loglog(t_ref, 0.5 * t_ref**0.5, 'r--', alpha=0.4, label=r'$\sim t^{0.5}$ (ref)')
ax2.loglog(t_ref, 0.5 * t_ref**1.0, 'g--', alpha=0.4, label=r'$\sim t^{1.0}$ (ref)')
ax2.loglog(t_ref, 0.5 * t_ref**1.5, 'b--', alpha=0.4, label=r'$\sim t^{1.5}$ (ref)')

ax2.set_xlabel('Tiempo $t$')
ax2.set_ylabel(r'$\langle x^2(t) \rangle$')
ax2.set_title(r'MSD: $\langle x^2 \rangle \sim t^{2H}$')
ax2.legend(fontsize=8, loc='upper left')
ax2.grid(True, alpha=0.3, which='both')

# --- 2. CTRW (Continuous Time Random Walk) para subdifusión ---
# Los tiempos de espera siguen distribución de ley de potencia: ψ(t) ~ t^{-(1+α)}

ax3 = axes[1, 0]

def ctrw_simulation(n_jumps, alpha_wait, dt_step=1.0):
    """
    Simula CTRW con tiempos de espera de cola pesada.
    alpha_wait < 1: subdifusión
    alpha_wait = 1: difusión normal (exponencial)
    """
    # Tiempos de espera: distribución Pareto (cola pesada)
    if alpha_wait < 1:
        waiting_times = (np.random.pareto(alpha_wait, n_jumps) + 1) * dt_step
    else:
        waiting_times = np.random.exponential(dt_step, n_jumps)
    
    # Saltos: gaussianos estándar
    jumps = np.random.randn(n_jumps)
    
    # Construir trayectoria
    cumulative_time = np.cumsum(waiting_times)
    cumulative_pos = np.cumsum(jumps)
    
    return cumulative_time, cumulative_pos

alpha_values = [0.5, 0.75, 1.0]
colors_a = ['#e74c3c', '#f39c12', '#2ecc71']
labels_a = [r'$\alpha = 0.5$ (subdifusión fuerte)', 
            r'$\alpha = 0.75$ (subdifusión)', 
            r'$\alpha = 1.0$ (difusión normal)']

for alpha_w, color, label in zip(alpha_values, colors_a, labels_a):
    ct, cp = ctrw_simulation(500, alpha_w, dt_step=0.1)
    # Truncar a un tiempo máximo razonable
    mask = ct < 100
    ax3.plot(ct[mask], cp[mask], color=color, label=label, alpha=0.7, linewidth=1.0)

ax3.set_xlabel('Tiempo $t$')
ax3.set_ylabel('Posición $x(t)$')
ax3.set_title('CTRW: Caminata Aleatoria de Tiempo Continuo')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# --- 3. Vuelos de Lévy ---
ax4 = axes[1, 1]

def levy_flight_2d(n_steps, alpha_levy):
    """
    Simula vuelo de Lévy en 2D.
    Los tamaños de paso siguen distribución estable con índice alpha_levy.
    alpha_levy = 2: Gaussiano (Browniano)
    alpha_levy < 2: colas pesadas (vuelos de Lévy)
    """
    # Aproximación: usar distribución Pareto para longitudes de paso
    if alpha_levy < 2:
        step_lengths = (np.random.pareto(alpha_levy, n_steps) + 1) * 0.1
    else:
        step_lengths = np.abs(np.random.randn(n_steps)) * 0.5
    
    angles = np.random.uniform(0, 2*np.pi, n_steps)
    dx = step_lengths * np.cos(angles)
    dy = step_lengths * np.sin(angles)
    
    x = np.cumsum(dx)
    y = np.cumsum(dy)
    
    return x, y

levy_alphas = [1.0, 1.5, 2.0]
colors_l = ['#e74c3c', '#9b59b6', '#3498db']
labels_l = [r'$\alpha_L = 1.0$ (Cauchy)', 
            r'$\alpha_L = 1.5$ (Lévy)', 
            r'$\alpha_L = 2.0$ (Gaussiano)']

for al, color, label in zip(levy_alphas, colors_l, labels_l):
    x_l, y_l = levy_flight_2d(1000, al)
    ax4.plot(x_l, y_l, color=color, label=label, alpha=0.6, linewidth=0.8)
    ax4.plot(x_l[0], y_l[0], 'o', color=color, markersize=6)
    ax4.plot(x_l[-1], y_l[-1], 's', color=color, markersize=6)

ax4.set_xlabel('$x$')
ax4.set_ylabel('$y$')
ax4.set_title('Vuelos de Lévy en 2D')
ax4.legend(fontsize=9)
ax4.set_aspect('equal')
ax4.grid(True, alpha=0.3)

plt.suptitle('Difusión Anómala: Extensiones del Movimiento Browniano', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- 4. Exponente de difusión vs parámetro de Hurst ---
fig2, ax = plt.subplots(figsize=(10, 6))

H_range = np.linspace(0.1, 0.9, 50)
alpha_from_H = 2 * H_range  # Para fBm: ⟨x²⟩ ~ t^{2H}

ax.plot(H_range, alpha_from_H, 'b-', linewidth=2.5, label=r'fBm: $\alpha = 2H$')
ax.axhline(y=1, color='green', linestyle='--', linewidth=1.5, label=r'Difusión normal ($\alpha = 1$)')
ax.axvline(x=0.5, color='green', linestyle=':', alpha=0.5)

ax.fill_between(H_range[H_range < 0.5], 0, alpha_from_H[H_range < 0.5], 
                alpha=0.15, color='red', label='Subdifusión')
ax.fill_between(H_range[H_range > 0.5], alpha_from_H[H_range > 0.5], 2, 
                alpha=0.15, color='blue', label='Superdifusión')

ax.set_xlabel('Parámetro de Hurst $H$', fontsize=13)
ax.set_ylabel(r'Exponente de difusión $\alpha$', fontsize=13)
ax.set_title(r'Clasificación de Difusión: $\langle x^2 \rangle \sim t^\alpha$', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 2)
ax.grid(True, alpha=0.3)

# Anotaciones
ax.annotate('Subdifusión\n(confinamiento,\nobstáculos)', 
            xy=(0.25, 0.5), fontsize=11, ha='center', color='red', fontweight='bold')
ax.annotate('Superdifusión\n(correlaciones\npersistentes)', 
            xy=(0.75, 1.5), fontsize=11, ha='center', color='blue', fontweight='bold')

plt.tight_layout()
plt.show()

print("=" * 70)
print("CLASIFICACIÓN DE REGÍMENES DE DIFUSIÓN")
print("=" * 70)
print(f"{'Régimen':<25} {'Exponente α':<20} {'Ejemplo físico'}")
print("-" * 70)
print(f"{'Subdifusión':<25} {'0 < α < 1':<20} {'Proteínas en membranas'}")
print(f"{'Difusión normal':<25} {'α = 1':<20} {'Partícula en fluido'}")
print(f"{'Superdifusión':<25} {'1 < α < 2':<20} {'Transporte activo'}")
print(f"{'Difusión balística':<25} {'α = 2':<20} {'Partícula libre'}")
print("=" * 70)

## 📚 7. Resumen y Referencias

### Resumen de Resultados Fundamentales

En este cuaderno hemos recorrido la teoría del **movimiento browniano** desde sus fundamentos históricos hasta las extensiones modernas:

| Concepto | Resultado Clave | Ecuación |
|----------|----------------|----------|
| **Einstein (1905)** | Desplazamiento cuadrático medio | $\langle x^2 \rangle = 2Dt$ |
| **Relación Stokes-Einstein** | Coeficiente de difusión | $D = \frac{k_B T}{6\pi\eta a}$ |
| **Ecuación de Langevin** | Dinámica estocástica | $m\dot{v} = -\gamma v + \eta(t)$ |
| **Teorema fluctuación-disipación** | Conexión ruido-fricción | $\langle \eta(t)\eta(t') \rangle = 2\gamma k_B T \,\delta(t-t')$ |
| **Ecuación de Fokker-Planck** | Evolución de densidad | $\frac{\partial P}{\partial t} = D\frac{\partial^2 P}{\partial x^2}$ |
| **Tiempos de primer paso** | Tiempo medio de escape | $\langle T \rangle = \frac{L^2}{2D}$ (desde el centro) |
| **Proceso Ornstein-Uhlenbeck** | Reversión a la media | $dv = -\frac{\gamma}{m}v\,dt + \sqrt{\frac{2\gamma k_B T}{m^2}}\,dW$ |
| **Difusión anómala** | MSD generalizado | $\langle x^2 \rangle \sim t^\alpha$ |

### Conexiones con Otras Áreas de la Física

1. **Mecánica estadística**: El movimiento browniano es una manifestación macroscópica de las fluctuaciones térmicas, proporcionando evidencia directa de la teoría molecular.

2. **Teoría de campos cuánticos**: La integral de camino de Feynman es formalmente análoga al proceso de Wiener mediante rotación de Wick ($t \to -i\tau$).

3. **Física financiera**: El modelo de Black-Scholes para valoración de opciones se basa en el movimiento browniano geométrico.

4. **Biofísica**: El transporte intracelular, la difusión de proteínas en membranas y la dinámica de polímeros son descritos por variantes del movimiento browniano.

5. **Cosmología**: Las fluctuaciones primordiales del universo temprano se modelan como campos estocásticos gaussianos.

---

### Referencias

#### Textos Clásicos
1. **Einstein, A.** (1905). *Über die von der molekularkinetischen Theorie der Wärme geforderte Bewegung von in ruhenden Flüssigkeiten suspendierten Teilchen*. Annalen der Physik, 322(8), 549-560.
2. **Langevin, P.** (1908). *Sur la théorie du mouvement brownien*. Comptes Rendus de l'Académie des Sciences, 146, 530-533.
3. **Perrin, J.** (1909). *Mouvement brownien et réalité moléculaire*. Annales de Chimie et de Physique, 18, 5-114.

#### Textos Modernos
4. **Van Kampen, N. G.** (2007). *Stochastic Processes in Physics and Chemistry*. 3ra edición, North-Holland.
5. **Gardiner, C. W.** (2009). *Stochastic Methods: A Handbook for the Natural and Social Sciences*. 4ta edición, Springer.
6. **Risken, H.** (1996). *The Fokker-Planck Equation: Methods of Solution and Applications*. 2da edición, Springer.
7. **Chandrasekhar, S.** (1943). *Stochastic Problems in Physics and Astronomy*. Reviews of Modern Physics, 15(1), 1-89.
8. **Uhlenbeck, G. E. & Ornstein, L. S.** (1930). *On the Theory of the Brownian Motion*. Physical Review, 36(5), 823-841.

#### Difusión Anómala
9. **Metzler, R. & Klafter, J.** (2000). *The random walk's guide to anomalous diffusion: a fractional dynamics approach*. Physics Reports, 339(1), 1-77.
10. **Bouchaud, J.-P. & Georges, A.** (1990). *Anomalous diffusion in disordered media*. Physics Reports, 195(4-5), 127-293.

---

*Cuaderno creado con fines didácticos. El tratamiento sigue el nivel de rigor de los textos de Gardiner y Van Kampen, con simulaciones numéricas para ilustrar los conceptos teóricos.*